In [ ]:






get_ipython().getoutput("pip install plyfile")


get_ipython().getoutput("pip install plyfile tqdm open3d -q")

import os, glob, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from plyfile import PlyData
from tqdm import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import open3d as o3d
warnings.filterwarnings('ignore')

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)





base_path = '/kaggle/input/datasets/haquanghung/rice-3d-segmentation-dataset/Label_Pot_Plant'
all_ply_files = glob.glob(os.path.join(base_path, '**', '*.ply'), recursive=True)
random.shuffle(all_ply_files)



### Chẩn đoán nếu không tìm thấy file


In [ ]:
if len(all_ply_files) == 0:
    import os
    print('KHÔNG TÌM THẤY FILE .ply!')
    print(f'base_path tồn tại: {os.path.exists(base_path)}')
    # Thử tìm đường dẫn đúng
    root = '/kaggle/input'
    for dirpath, dirnames, filenames in os.walk(root):
        plys = [f for f in filenames if f.endswith('.ply')]
        if plys:
            print(f'  Tim thay {len(plys)} file .ply tai: {dirpath}')
            if len(plys) <= 3:
                for f in plys: print(f'    {f}')
            break
    raise FileNotFoundError(f'Không có file .ply tại: {base_path}')

print(f'Tổng file .ply: {len(all_ply_files)}')
print(f'Ví dụ: {all_ply_files[0]}')



### Quét phân bố class


In [ ]:
print('\nĐang quét toàn bộ dataset...')
n_bg = n_pot = n_plant = 0
files_with_bg = 0

for fp in tqdm(all_ply_files, desc='Scanning'):
    try:
        v     = PlyData.read(fp)['vertex']
        props = {p.name for p in v.properties}
        pot   = np.array(v['scalar_Pot'],   dtype=np.float32) if 'scalar_Pot'   in props else np.full(len(v), np.nan)
        plant = np.array(v['scalar_Plant'], dtype=np.float32) if 'scalar_Plant' in props else np.full(len(v), np.nan)
        cnt_pot   = int((~np.isnan(pot)).sum())
        cnt_plant = int((~np.isnan(plant)).sum())
        cnt_bg    = len(v) - cnt_pot - cnt_plant
        n_pot   += cnt_pot
        n_plant += cnt_plant
        n_bg    += max(cnt_bg, 0)
        if cnt_bg > 0: files_with_bg += 1
    except Exception as e:
        print(f'  Lỗi: {os.path.basename(fp)} — {e}')

total = n_bg + n_pot + n_plant
if total == 0:
    raise ValueError('Không đọc được điểm nào! Kiểm tra lại cấu trúc file .ply.')

print(f'\n{"="*48}')
print(f'{"Class":<22} {"Điểm":>12} {"Tỉ lệ":>8}')
print(f'{"-"*48}')
print(f'{"Background":<22} {n_bg:>12,} {n_bg/total*100:>7.1f}%')
print(f'{"Pot/Chậu":<22} {n_pot:>12,} {n_pot/total*100:>7.1f}%')
print(f'{"Plant/Cây":<22} {n_plant:>12,} {n_plant/total*100:>7.1f}%')
print(f'{"TOTAL":<22} {total:>12,}')
print(f'\nFile có Background: {files_with_bg} / {len(all_ply_files)}')
print(f'{"="*48}')



### Chia train/test 80/20


In [ ]:
split = int(len(all_ply_files) * 0.8)
train_files = all_ply_files[:split]
test_files  = all_ply_files[split:]
print(f'\nTrain: {len(train_files)} | Test: {len(test_files)}')



### Gợi ý NUM_CLASSES


In [ ]:
if files_with_bg == 0:
    print('\n=> Không có Background → NUM_CLASSES = 2  (0=Pot, 1=Plant)')
else:
    pct = files_with_bg / len(all_ply_files) * 100
    print(f'\n=> {files_with_bg} file ({pct:.1f}%) có Background.')
    if pct > 20:
        print('   Background chiếm đáng kể → nên dùng NUM_CLASSES = 3')
        print('   Đổi NUM_CLASSES = 3 ở Cell 2.')
    else:
        print('   Background rất ít → có thể giữ NUM_CLASSES = 2 hoặc đổi lên 3.')






# ════════════════════════════════════════
#  Đọc file PLY → dict numpy
# ════════════════════════════════════════
def read_ply(file_path):
    ply   = PlyData.read(file_path)
    v     = ply['vertex']
    props = {p.name for p in v.properties}

    xyz = np.vstack([v['x'], v['y'], v['z']]).T.astype(np.float32)  # (N,3)

    # ── Label 2 class: Pot=0, Plant=1 ──
    label = np.zeros(len(v), dtype=np.int64)   # default: Pot=0
    if 'scalar_Plant' in props:
        plant = np.array(v['scalar_Plant'], dtype=np.float32)
        label[~np.isnan(plant)] = 1            # Plant = 1

    # ── Normal (nếu file đã có sẵn) ──
    if all(c in props for c in ['nx', 'ny', 'nz']):
        normal = np.vstack([v['nx'], v['ny'], v['nz']]).T.astype(np.float32)
    else:
        normal = None

    return {'xyz': xyz, 'label': label, 'normal': normal}


# ════════════════════════════════════════
#  Precompute & cache normal vector
#  Tính 1 lần cho toàn bộ dataset,
#  lưu vào dict trong RAM.
#  Tránh gọi Open3D lặp lại mỗi epoch.
# ════════════════════════════════════════
def precompute_normals(file_paths, k=20):
    """
    Đọc tất cả file, tính normal bằng Open3D, cache vào dict.
    Chỉ chạy 1 lần trước khi train.
    Trả về: {file_path: np.ndarray (N, 3)}
    """
    cache = {}
    for fp in tqdm(file_paths, desc='Precompute normals'):
        try:
            data = read_ply(fp)
            xyz  = data['xyz']
            if data['normal'] is not None:
                # File đã có normal sẵn → dùng luôn
                cache[fp] = data['normal']
            else:
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(xyz.astype(np.float64))
                pcd.estimate_normals(
                    search_param=o3d.geometry.KDTreeSearchParamKNN(knn=k))
                pcd.orient_normals_to_align_with_direction([0, 0, 1])
                normals = np.asarray(pcd.normals).astype(np.float32)
                cache[fp] = np.nan_to_num(normals, nan=0.0)
        except Exception as e:
            print(f'  Normal error {os.path.basename(fp)}: {e}')
            # Fallback: normal = [0,0,1] cho tất cả điểm
            n = len(PlyData.read(fp)['vertex'])
            cache[fp] = np.tile([0., 0., 1.], (n, 1)).astype(np.float32)
    print(f'Normal cache: {len(cache)} files')
    return cache


# ════════════════════════════════════════
#  Dataset
# ════════════════════════════════════════
class RicePointCloudDataset(Dataset):
    INPUT_DIM = {'xyz': 3, 'xyz_h': 4, 'xyz_h_n': 7}

    def __init__(self, file_paths, num_points=4096,
                 input_mode='xyz_h_n', augment=False,
                 normal_cache=None):
        """
        normal_cache: dict {file_path: np.ndarray (N,3)}
                      truyền vào từ precompute_normals().
                      Nếu None và mode='xyz_h_n' → tính on-the-fly (chậm).
        """
        self.file_paths   = file_paths
        self.num_points   = num_points
        self.input_mode   = input_mode
        self.augment      = augment
        self.normal_cache = normal_cache or {}
        assert input_mode in self.INPUT_DIM

    @property
    def in_channels(self):
        return self.INPUT_DIM[self.input_mode]

    def __len__(self):
        return len(self.file_paths)

    def _augment(self, xyz):
        xyz = xyz + np.random.normal(0, 0.005, xyz.shape).astype(np.float32)
        theta = np.random.uniform(0, 2 * np.pi)
        c, s  = np.cos(theta), np.sin(theta)
        Rz = np.array([[c,-s,0],[s,c,0],[0,0,1]], dtype=np.float32)
        return xyz @ Rz.T * np.random.uniform(0.85, 1.15)

    def __getitem__(self, idx):
        fp     = self.file_paths[idx]
        data   = read_ply(fp)
        xyz    = data['xyz']
        labels = data['label']
        N      = xyz.shape[0]

        # ── Sampling ──
        choices = np.random.choice(N, self.num_points,
                                   replace=(N < self.num_points))
        xyz    = xyz[choices]
        labels = labels[choices]

        # ── Normalize XYZ ──
        xyz -= xyz.mean(0)
        d    = np.max(np.linalg.norm(xyz, axis=1))
        if d > 0: xyz /= d

        if self.augment:
            xyz = self._augment(xyz)

        # ── Relative height ──
        rel_h = (xyz[:, 2] - xyz[:, 2].min()).reshape(-1, 1)

        # ── Normal từ cache (đã precompute) ──
        if self.input_mode == 'xyz_h_n':
            if fp in self.normal_cache:
                normal = self.normal_cache[fp][choices]
                # Nếu augment có rotation → rotate normal theo
                if self.augment:
                    theta = np.random.uniform(0, 2 * np.pi)
                    c, s  = np.cos(theta), np.sin(theta)
                    Rz    = np.array([[c,-s,0],[s,c,0],[0,0,1]], dtype=np.float32)
                    normal = normal @ Rz.T
            else:
                # Fallback on-the-fly (hiếm xảy ra)
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(xyz.astype(np.float64))
                pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(20))
                pcd.orient_normals_to_align_with_direction([0, 0, 1])
                normal = np.nan_to_num(
                    np.asarray(pcd.normals).astype(np.float32), nan=0.0)

        # ── Ghép features ──
        if self.input_mode == 'xyz':
            feat = xyz
        elif self.input_mode == 'xyz_h':
            feat = np.concatenate([xyz, rel_h], axis=1)
        elif self.input_mode == 'xyz_h_n':
            feat = np.concatenate([xyz, rel_h, normal], axis=1)

        return (torch.from_numpy(feat).float().T,
                torch.from_numpy(labels).long())


# ════════════════════════════════════════
#  Config & DataLoader
# ════════════════════════════════════════
NUM_POINTS  = 4096
BATCH_SIZE  = 8
NUM_CLASSES = 2          # 0=Pot/Chậu  1=Plant/Cây lúa
# Đổi thành 3 nếu Cell 1 quét ra có Background
INPUT_MODE  = 'xyz_h_n'  # geometry-only 7D
NORMAL_K    = 20         # kNN để tính normal



### Precompute normals 1 lần cho toàn dataset


In [ ]:
print('Precomputing normals (chỉ chạy 1 lần)...')
all_files_for_normal = list(set(train_files + test_files))
normal_cache = precompute_normals(all_files_for_normal, k=NORMAL_K)



### Build Dataset


In [ ]:
train_dataset = RicePointCloudDataset(
    train_files, NUM_POINTS, INPUT_MODE, augment=True,  normal_cache=normal_cache)
test_dataset  = RicePointCloudDataset(
    test_files,  NUM_POINTS, INPUT_MODE, augment=False, normal_cache=normal_cache)
IN_CH = train_dataset.in_channels

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

cls_names = {2:'0=Pot 1=Plant', 3:'0=BG 1=Pot 2=Plant'}.get(NUM_CLASSES,'')
print(f'Input mode : {INPUT_MODE}  ({IN_CH} channels)')
print(f'Num classes: {NUM_CLASSES}  ({cls_names})')
print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')

pts, lbs = next(iter(train_loader))
print(f'Batch shape: pts={tuple(pts.shape)}  labels={tuple(lbs.shape)}')
print(f'Label values: {lbs.unique().tolist()}')






# ════════════════════════════════════════
#  1. PointNet (baseline tham chiếu)
# ════════════════════════════════════════
class PointNetSeg(nn.Module):
    def __init__(self, num_classes=3, in_channel=7):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channel, 64,   1); self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64,         128,  1); self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128,        1024, 1); self.bn3 = nn.BatchNorm1d(1024)
        self.conv4 = nn.Conv1d(1024+64,    512,  1); self.bn4 = nn.BatchNorm1d(512)
        self.conv5 = nn.Conv1d(512,        256,  1); self.bn5 = nn.BatchNorm1d(256)
        self.conv6 = nn.Conv1d(256,        128,  1); self.bn6 = nn.BatchNorm1d(128)
        self.conv7 = nn.Conv1d(128, num_classes, 1)

    def forward(self, x):                      # (B, C, N)
        n  = x.size(2)
        x1 = F.relu(self.bn1(self.conv1(x)))
        x2 = F.relu(self.bn2(self.conv2(x1)))
        x3 = self.bn3(self.conv3(x2))
        gf = x3.max(2, keepdim=True)[0].expand(-1, -1, n)
        x  = torch.cat([x1, gf], 1)
        x  = F.relu(self.bn4(self.conv4(x)))
        x  = F.relu(self.bn5(self.conv5(x)))
        x  = F.relu(self.bn6(self.conv6(x)))
        return F.log_softmax(self.conv7(x).transpose(2, 1), dim=-1)


# ════════════════════════════════════════
#  2. PointNet++ — ưu tiên #1
#     Ball Query học vùng chuyển tiếp
#     Pot↔Plant cục bộ rất hiệu quả
# ════════════════════════════════════════
def sq_dist(src, dst):
    B,N,_ = src.shape; _,M,_ = dst.shape
    d = -2*torch.bmm(src, dst.permute(0,2,1))
    d += (src**2).sum(-1,keepdim=True)
    d += (dst**2).sum(-1).unsqueeze(1)
    return d

def idx_pts(pts, idx):
    B = pts.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0]  = 1
    bi = torch.arange(B,device=pts.device).view(vs).repeat(rs)
    return pts[bi, idx]

def fps(xyz, npt):
    B,N,_ = xyz.shape
    ctr  = torch.zeros(B,npt,dtype=torch.long,device=xyz.device)
    dist = torch.ones(B,N,device=xyz.device)*1e10
    far  = torch.randint(0,N,(B,),device=xyz.device)
    bi   = torch.arange(B,device=xyz.device)
    for i in range(npt):
        ctr[:,i] = far
        c    = xyz[bi,far].unsqueeze(1)
        dist = torch.min(dist, ((xyz-c)**2).sum(-1))
        far  = dist.argmax(-1)
    return ctr

def ball_q(r, K, xyz, nxyz):
    sq  = sq_dist(nxyz, xyz)
    idx = sq.argsort(-1)[:,:,:K]
    gf  = idx[:,:,0:1].expand_as(idx)
    idx[sq.sort(-1)[0][:,:,:K] > r**2] = gf[sq.sort(-1)[0][:,:,:K] > r**2]
    return idx

class SA(nn.Module):
    """Set Abstraction — học đặc trưng cục bộ theo ball query."""
    def __init__(self, npt, r, K, in_ch, dims):
        super().__init__()
        self.npt=npt; self.r=r; self.K=K
        layers=[]; last=in_ch+3
        for d in dims:
            layers+=[nn.Conv2d(last,d,1),nn.BatchNorm2d(d),nn.ReLU(True)]; last=d
        self.mlp=nn.Sequential(*layers)
    def forward(self, xyz, pts):
        fi  = fps(xyz, self.npt)
        nxyz= idx_pts(xyz, fi)
        idx = ball_q(self.r, self.K, xyz, nxyz)
        gxyz= idx_pts(xyz, idx) - nxyz.unsqueeze(2)
        gp  = torch.cat([gxyz, idx_pts(pts,idx)],dim=-1) if pts is not None else gxyz
        out = self.mlp(gp.permute(0,3,2,1)).max(2)[0].permute(0,2,1)
        return nxyz, out

class FP(nn.Module):
    """Feature Propagation — upsample bằng nội suy IDW."""
    def __init__(self, in_ch, dims):
        super().__init__()
        layers=[]; last=in_ch
        for d in dims:
            layers+=[nn.Conv1d(last,d,1),nn.BatchNorm1d(d),nn.ReLU(True)]; last=d
        self.mlp=nn.Sequential(*layers)
    def forward(self, xyz1, xyz2, p1, p2):
        B,N,_ = xyz1.shape; _,S,_ = xyz2.shape
        if S==1:
            ip = p2.expand(B,N,-1)
        else:
            d,idx = sq_dist(xyz1,xyz2).sort(-1)
            d,idx = d[:,:,:3], idx[:,:,:3]
            w = 1./(d+1e-8); w = w/w.sum(-1,keepdim=True)
            ip = (idx_pts(p2,idx)*w.unsqueeze(-1)).sum(2)
        f = torch.cat([p1,ip],dim=-1) if p1 is not None else ip
        return self.mlp(f.permute(0,2,1)).permute(0,2,1)

class PointNetPPSeg(nn.Module):
    def __init__(self, num_classes=3, in_channel=7):
        super().__init__()
        feat_ch = in_channel - 3   # channels ngoài xyz (rel_h + normal = 4)
        self.sa1 = SA(1024, 0.1, 32, feat_ch,   [32, 32, 64])
        self.sa2 = SA(256,  0.2, 32, 64,         [64, 64, 128])
        self.sa3 = SA(64,   0.4, 32, 128,        [128, 128, 256])
        self.fp3 = FP(256+128,       [256, 128])
        self.fp2 = FP(128+64,        [128, 64])
        self.fp1 = FP(64+feat_ch,    [64, 64])
        self.head= nn.Sequential(
            nn.Conv1d(64, 64, 1), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Dropout(0.5), nn.Conv1d(64, num_classes, 1))
        self.feat_ch = feat_ch

    def forward(self, x):                    # (B, C, N)
        xyz  = x[:, :3, :].permute(0,2,1)   # (B,N,3)
        pts0 = x[:, 3:, :].permute(0,2,1) if self.feat_ch > 0 else None
        xyz1,p1 = self.sa1(xyz,  pts0)
        xyz2,p2 = self.sa2(xyz1, p1)
        xyz3,p3 = self.sa3(xyz2, p2)
        p2  = self.fp3(xyz2, xyz3, p2, p3)
        p1  = self.fp2(xyz1, xyz2, p1, p2)
        p0  = self.fp1(xyz,  xyz1, pts0, p1)
        out = self.head(p0.permute(0,2,1))
        return F.log_softmax(out.permute(0,2,1), dim=-1)


# ════════════════════════════════════════
#  3. KPConv (simplified) — ưu tiên #2
#     Kernel điểm dạng cầu 3D
#     Học hình dạng thành chậu cong
# ════════════════════════════════════════
class KPConvLayer(nn.Module):
    """KPConv đơn giản hóa: K kernel points trên sphere, aggregate bằng distance-weighted sum."""
    def __init__(self, in_ch, out_ch, K=15, radius=0.1):
        super().__init__()
        self.K      = K
        self.radius = radius
        # Kernel points: phân bố đều trên sphere đơn vị
        pts = []
        for i in range(K):
            theta = np.arccos(1 - 2*(i+0.5)/K)
            phi   = np.pi * (1 + 5**0.5) * i
            pts.append([np.sin(theta)*np.cos(phi),
                        np.sin(theta)*np.sin(phi),
                        np.cos(theta)])
        kpts = torch.tensor(pts, dtype=torch.float32)   # (K,3)
        self.register_buffer('kernel_pts', kpts)

        self.W   = nn.Parameter(torch.randn(K, in_ch, out_ch) * 0.01)
        self.bn  = nn.BatchNorm1d(out_ch)

    def forward(self, xyz, feats, new_xyz=None):
        """
        xyz:     (B, N, 3)
        feats:   (B, N, C_in)
        new_xyz: (B, M, 3) — nếu None thì dùng xyz (no downsampling)
        """
        if new_xyz is None:
            new_xyz = xyz
        B, N, _ = xyz.shape
        _, M, _ = new_xyz.shape
        C_in    = feats.shape[-1]

        # Lấy K điểm lân cận cho mỗi điểm trong new_xyz
        dists = sq_dist(new_xyz, xyz)         # (B, M, N)
        _, nn_idx = dists.topk(min(16, N), largest=False, dim=-1)  # (B, M, 16)
        K_nb = nn_idx.shape[-1]

        nb_xyz  = idx_pts(xyz,   nn_idx)      # (B, M, K_nb, 3)
        nb_feat = idx_pts(feats, nn_idx)      # (B, M, K_nb, C_in)

        # Relative position vs kernel points
        rel = nb_xyz - new_xyz.unsqueeze(2)   # (B, M, K_nb, 3)
        # Distance to each kernel point
        kp  = self.kernel_pts.unsqueeze(0).unsqueeze(0).unsqueeze(0)  # (1,1,1,K,3)
        rel_e = rel.unsqueeze(3)                                        # (B,M,K_nb,1,3)
        kp_dist = ((rel_e - kp * self.radius)**2).sum(-1)             # (B,M,K_nb,K)

        # Kornel weights: linear correlation
        sigma   = self.radius
        w       = torch.clamp(1 - kp_dist.sqrt() / sigma, min=0)      # (B,M,K_nb,K)

        # Aggregate: sum over neighbors, weighted by kernel
        # nb_feat: (B,M,K_nb,C_in), w: (B,M,K_nb,K)
        # → (B,M,K,C_in)
        agg = torch.einsum('bmkc,bmkK->bmKc', nb_feat, w)             # (B,M,K,C_in)

        # Apply kernel weights W: (K, C_in, C_out)
        out = torch.einsum('bmkc,kco->bmo', agg, self.W)              # (B,M,C_out)
        out = self.bn(out.reshape(B*M, -1)).reshape(B, M, -1)
        return F.relu(out)


class KPConvSeg(nn.Module):
    def __init__(self, num_classes=3, in_channel=7):
        super().__init__()
        self.enc1 = KPConvLayer(in_channel, 32,  K=15, radius=0.10)
        self.enc2 = KPConvLayer(32,         64,  K=15, radius=0.20)
        self.enc3 = KPConvLayer(64,         128, K=15, radius=0.40)

        self.dec3 = nn.Sequential(nn.Linear(128+64, 64), nn.ReLU())
        self.dec2 = nn.Sequential(nn.Linear(64+32,  32), nn.ReLU())
        self.dec1 = nn.Sequential(nn.Linear(32+in_channel, 32), nn.ReLU())

        self.head = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, num_classes))

    @staticmethod
    def _upsample(new_xyz, old_xyz, new_feat):
        """Nearest-neighbor upsample."""
        d   = sq_dist(old_xyz, new_xyz)    # (B, N_old, N_new)
        idx = d.argmin(-1)                  # (B, N_old)
        return idx_pts(new_feat, idx)       # (B, N_old, C)

    def forward(self, x):               # (B, C, N)
        xyz0  = x[:, :3, :].permute(0,2,1)      # (B,N,3)
        feat0 = x.permute(0,2,1)                 # (B,N,C)
        B, N, _ = xyz0.shape

        # Encoder (với FPS downsampling)
        idx1 = fps(xyz0, N//4);    xyz1 = idx_pts(xyz0, idx1)
        f1   = self.enc1(xyz0, feat0, new_xyz=xyz1)

        idx2 = fps(xyz1, N//16);   xyz2 = idx_pts(xyz1, idx2)
        f2   = self.enc2(xyz1, f1,  new_xyz=xyz2)

        idx3 = fps(xyz2, N//64);   xyz3 = idx_pts(xyz2, idx3)
        f3   = self.enc3(xyz2, f2,  new_xyz=xyz3)

        # Decoder
        up3 = self._upsample(xyz3, xyz2, f3)
        d3  = self.dec3(torch.cat([up3, f2], dim=-1))

        up2 = self._upsample(xyz2, xyz1, d3)
        d2  = self.dec2(torch.cat([up2, f1], dim=-1))

        up1 = self._upsample(xyz1, xyz0, d2)
        d1  = self.dec1(torch.cat([up1, feat0], dim=-1))

        out = self.head(d1)              # (B,N,C)
        return F.log_softmax(out, dim=-1)


# ════════════════════════════════════════
#  4. DGCNN — ưu tiên #3
#     kNN graph động — density 20x giữa
#     Pot và Plant giúp graph tự phân tầng
# ════════════════════════════════════════
def knn_graph(x, k):
    B,C,N = x.shape
    inner = -2*torch.bmm(x.permute(0,2,1), x)
    sq    = (x**2).sum(1, keepdim=True)
    dist  = inner + sq + sq.permute(0,2,1)
    return dist.topk(k+1, largest=False)[1][:, :, 1:]

def edge_feat(x, k):
    B, C, N = x.shape
    idx = knn_graph(x, k)                          # (B, N, k)
    xt  = x.permute(0, 2, 1).contiguous()          # (B, N, C)
    bi  = torch.arange(B, device=x.device).view(B,1,1).expand(B, N, k)        # (B, N, k)
    # flatten cả hai về 1D rồi index, sau đó reshape lại
    nb  = xt[bi.reshape(-1), idx.reshape(-1)].reshape(B, N, k, C).permute(0, 3, 1, 2)                   # (B, C, N, k)
    ctr = x.unsqueeze(-1).expand_as(nb)
    return torch.cat([ctr, nb - ctr], dim=1)       # (B, 2C, N, k)

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=20):
        super().__init__()
        self.k = k
        self.net = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True))
    def forward(self, x):
        return self.net(edge_feat(x, self.k)).max(-1)[0]

class DGCNNSeg(nn.Module):
    def __init__(self, num_classes=3, in_channel=7, k=20):
        super().__init__()
        self.ec1 = EdgeConv(in_channel, 64,  k)
        self.ec2 = EdgeConv(64,         64,  k)
        self.ec3 = EdgeConv(64,         128, k)
        self.ec4 = EdgeConv(128,        256, k)
        self.cg  = nn.Sequential(nn.Conv1d(512, 1024, 1),
                                  nn.BatchNorm1d(1024), nn.LeakyReLU(0.2))
        self.seg = nn.Sequential(
            nn.Conv1d(1536, 512, 1), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Conv1d(512,  256, 1), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Dropout(0.5), nn.Conv1d(256, num_classes, 1))

    def forward(self, x):
        x1=self.ec1(x); x2=self.ec2(x1); x3=self.ec3(x2); x4=self.ec4(x3)
        lc = torch.cat([x1,x2,x3,x4], 1)                   # (B,512,N)
        gf = self.cg(lc).max(-1, keepdim=True)[0].expand_as(self.cg(lc))
        return F.log_softmax(
            self.seg(torch.cat([lc, gf], 1)).permute(0,2,1), dim=-1)


# ════════════════════════════════════════
#  5. PointNeXt — nâng cấp PointNet++ (ECCV 2022)
#     Inverted Residual MLP + skip connection
# ════════════════════════════════════════
class InvertedResMLP(nn.Module):
    """Inverted Residual block: expand → depthwise → project + skip."""
    def __init__(self, in_ch, expansion=4):
        super().__init__()
        mid = in_ch * expansion
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, mid, 1, bias=False), nn.BatchNorm1d(mid), nn.ReLU(True),
            nn.Conv1d(mid, mid, 1, bias=False, groups=mid),
            nn.BatchNorm1d(mid), nn.ReLU(True),
            nn.Conv1d(mid, in_ch, 1, bias=False), nn.BatchNorm1d(in_ch))
    def forward(self, x):
        return F.relu(x + self.net(x), inplace=True)


class SA_NeXt(nn.Module):
    def __init__(self, npt, r, K, in_ch, out_ch):
        super().__init__()
        self.npt, self.r, self.K = npt, r, K
        self.lifting = nn.Sequential(
            nn.Conv2d(in_ch + 3, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.res = InvertedResMLP(out_ch, expansion=4)

    def forward(self, xyz, pts):
        fi    = fps(xyz, self.npt)
        nxyz  = idx_pts(xyz, fi)
        idx   = ball_q(self.r, self.K, xyz, nxyz)
        gxyz  = idx_pts(xyz, idx) - nxyz.unsqueeze(2)
        gp    = torch.cat([gxyz, idx_pts(pts, idx)], dim=-1) if pts is not None else gxyz
        gp    = gp.permute(0, 3, 2, 1)
        gp    = self.lifting(gp).max(2)[0]
        gp    = self.res(gp)
        return nxyz, gp.permute(0, 2, 1)


class PointNeXtSeg(nn.Module):
    def __init__(self, num_classes=2, in_channel=3):
        super().__init__()
        self.stem = nn.Conv1d(in_channel, 32, 1)
        self.sa1 = SA_NeXt(1024, 0.10, 32, 32,  64)
        self.sa2 = SA_NeXt(256,  0.20, 32, 64,  128)
        self.sa3 = SA_NeXt(64,   0.40, 32, 128, 256)
        self.fp3 = FP(256+128, [256, 128])
        self.fp2 = FP(128+64,  [128, 64])
        self.fp1 = FP(64+32,   [64, 64])
        self.head = nn.Sequential(
            nn.Conv1d(64, 64, 1), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Dropout(0.5), nn.Conv1d(64, num_classes, 1))

    def forward(self, x):
        xyz   = x[:, :3, :].permute(0, 2, 1)
        f0    = self.stem(x).permute(0, 2, 1)
        xyz1, f1 = self.sa1(xyz,  f0)
        xyz2, f2 = self.sa2(xyz1, f1)
        xyz3, f3 = self.sa3(xyz2, f2)
        f2 = self.fp3(xyz2, xyz3, f2, f3)
        f1 = self.fp2(xyz1, xyz2, f1, f2)
        f0 = self.fp1(xyz,  xyz1, f0, f1)
        out = self.head(f0.permute(0, 2, 1))
        return F.log_softmax(out.permute(0, 2, 1), dim=-1)


# ════════════════════════════════════════
#  6. Point Transformer v1 (Zhao et al. 2021)
#     Vector self-attention trên kNN — memory-efficient
# ════════════════════════════════════════
class PointTransformerLayer(nn.Module):
    def __init__(self, dim, k=16):
        super().__init__()
        self.k = k
        self.q = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.pos_mlp = nn.Sequential(
            nn.Linear(3, dim), nn.ReLU(True), nn.Linear(dim, dim))
        self.attn_mlp = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(True), nn.Linear(dim, dim))

    def forward(self, xyz, feat):
        B, N, C = feat.shape
        k = min(self.k, N)
        d = sq_dist(xyz, xyz)
        _, idx = d.topk(k+1, largest=False)
        idx = idx[:, :, 1:]

        q = self.q(feat)
        kf = idx_pts(self.k_proj(feat), idx)
        v  = idx_pts(self.v(feat), idx)
        nb_xyz = idx_pts(xyz, idx)

        pos = self.pos_mlp(nb_xyz - xyz.unsqueeze(2))
        attn = self.attn_mlp(q.unsqueeze(2) - kf + pos)
        attn = F.softmax(attn / (C**0.5), dim=2)
        return (attn * (v + pos)).sum(dim=2)


class PointTransformerBlock(nn.Module):
    def __init__(self, dim, k=16):
        super().__init__()
        self.attn = PointTransformerLayer(dim, k)
        self.norm1 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
        self.norm2 = nn.LayerNorm(dim)
    def forward(self, xyz, feat):
        feat = feat + self.attn(xyz, self.norm1(feat))
        feat = feat + self.mlp(self.norm2(feat))
        return feat


class PointTransformerSeg(nn.Module):
    def __init__(self, num_classes=2, in_channel=3, k=16):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Conv1d(in_channel, 64, 1),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Conv1d(64, 128, 1),
            nn.BatchNorm1d(128), nn.ReLU(True))
        self.t1 = PointTransformerBlock(128, k)
        self.t2 = PointTransformerBlock(128, k)
        self.t3 = PointTransformerBlock(128, k)
        self.t4 = PointTransformerBlock(128, k)
        self.head = nn.Sequential(
            nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.Dropout(0.5), nn.Conv1d(256, num_classes, 1))

    def forward(self, x):
        xyz  = x[:, :3, :].permute(0, 2, 1)
        feat = self.embed(x).permute(0, 2, 1)
        feat = self.t1(xyz, feat)
        feat = self.t2(xyz, feat)
        feat = self.t3(xyz, feat)
        feat = self.t4(xyz, feat)
        out  = self.head(feat.permute(0, 2, 1))
        return F.log_softmax(out.permute(0, 2, 1), dim=-1)




### Smoke test


In [ ]:
dummy = torch.zeros(2, IN_CH, NUM_POINTS).to(device)
for name, cls, kw in [
    ('PointNet',     PointNetSeg,        {}),
    ('PointNet++',   PointNetPPSeg,      {}),
    ('KPConv',       KPConvSeg,          {}),
    ('DGCNN',        DGCNNSeg,           {'k': 20}),
    ('PointNeXt',    PointNeXtSeg,       {}),
    ('PointTrans',   PointTransformerSeg,{'k': 16}),
]:
    m   = cls(num_classes=NUM_CLASSES, in_channel=IN_CH, **kw).to(device)
    out = m(dummy)
    n   = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:12s}  in={tuple(dummy.shape)}  out={tuple(out.shape)}  params={n:,}')
del dummy






def compute_class_weights(file_list, num_classes, max_files=200):
    """
    Tính class weight bằng inverse frequency.
    Class ít điểm → weight cao hơn (model chú ý hơn).

    LƯU Ý: phải khớp với label mapping trong read_ply():
      NUM_CLASSES=2: class 0 = Pot+Background (gộp), class 1 = Plant
      NUM_CLASSES=3: class 0 = Background, class 1 = Pot, class 2 = Plant
    """
    counts = np.zeros(num_classes, dtype=np.float64)
    sample = file_list[:min(max_files, len(file_list))]

    for fp in tqdm(sample, desc='Computing class weights', leave=False):
        try:
            v = PlyData.read(fp)['vertex']
            props = {p.name for p in v.properties}
            pot   = np.array(v['scalar_Pot'],   dtype=np.float32) if 'scalar_Pot'   in props else np.full(len(v), np.nan)
            plant = np.array(v['scalar_Plant'], dtype=np.float32) if 'scalar_Plant' in props else np.full(len(v), np.nan)
            n_pot   = int((~np.isnan(pot)).sum())
            n_plant = int((~np.isnan(plant)).sum())
            n_bg    = len(v) - n_pot - n_plant

            if num_classes == 2:
                counts[0] += n_pot + n_bg   # Pot+BG → class 0
                counts[1] += n_plant        # Plant  → class 1
            else:
                counts[0] += n_bg
                counts[1] += n_pot
                counts[2] += n_plant
        except Exception:
            pass

    counts  = np.where(counts == 0, 1, counts)
    # Inverse frequency: class hiếm → weight cao
    weights = counts.sum() / (num_classes * counts)
    # Cap để tránh weight quá lớn gây nhiễu gradient
    weights = np.clip(weights, 0.5, 10.0)
    return torch.tensor(weights, dtype=torch.float32)


class_weights = compute_class_weights(train_files, NUM_CLASSES)
names_cls = ['Pot+BG', 'Plant'] if NUM_CLASSES == 2 else ['Background', 'Pot', 'Plant']
print('Class weights (inverse frequency, capped at [0.5, 10]):')
for c, (name, w) in enumerate(zip(names_cls, class_weights)):
    print(f'  class {c} ({name:12s}): {w:.4f}')



def compute_metrics(pred, target, num_classes):
    correct = (pred == target).sum().item()
    ious = []
    for c in range(num_classes):
        tp = ((pred == c) & (target == c)).sum().item()
        fp = ((pred == c) & (target != c)).sum().item()
        fn = ((pred != c) & (target == c)).sum().item()
        d  = tp + fp + fn
        ious.append(tp / d if d > 0 else float('nan'))
    return {
        'acc':  correct / target.numel(),
        'miou': float(np.nanmean(ious)),
        'iou_per_class': ious,
    }


def train_model(model, train_loader, test_loader,
                num_classes, class_weights,
                epochs=100, lr=1e-3,
                save_path='best.pth', model_name='Model'):

    criterion = nn.NLLLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # IoU theo num_classes thực tế (không hardcode 3)
    history = {
        'train_loss':    [], 'test_loss':     [],
        'train_acc':     [], 'test_acc':      [], 'test_miou': [],
        'iou_per_class': [[] for _ in range(num_classes)],
    }
    cls_labels = ['Pot+BG', 'Plant'] if num_classes == 2 else ['BG', 'Pot', 'Plant']
    best_miou  = 0.0

    for ep in range(1, epochs + 1):
        # ── Train ──
        model.train()
        tl = tc = tt = 0
        for pts, lbl in tqdm(train_loader, desc=f'[{model_name}] {ep}/{epochs}', leave=False):
            pts, lbl = pts.to(device), lbl.to(device)
            optimizer.zero_grad()
            out  = model(pts)
            loss = criterion(out.reshape(-1, num_classes), lbl.reshape(-1))
            loss.backward()
            optimizer.step()
            tl += loss.item()
            tc += (out.argmax(-1) == lbl).sum().item()
            tt += lbl.numel()
        scheduler.step()

        # ── Eval ──
        model.eval()
        el = 0; all_pred = []; all_gt = []
        with torch.no_grad():
            for pts, lbl in test_loader:
                pts, lbl = pts.to(device), lbl.to(device)
                out  = model(pts)
                el  += criterion(out.reshape(-1, num_classes), lbl.reshape(-1)).item()
                all_pred.append(out.argmax(-1).cpu())
                all_gt.append(lbl.cpu())

        P = torch.cat(all_pred); G = torch.cat(all_gt)
        m   = compute_metrics(P, G, num_classes)
        iou = m['iou_per_class']

        history['train_loss'].append(tl / len(train_loader))
        history['test_loss'].append(el  / len(test_loader))
        history['train_acc'].append(tc / tt)
        history['test_acc'].append(m['acc'])
        history['test_miou'].append(m['miou'])
        for ci in range(num_classes):
            v = iou[ci]
            history['iou_per_class'][ci].append(0.0 if np.isnan(v) else v)

        iou_str = ' | '.join(f'{cls_labels[ci]}={iou[ci]*100:.1f}%'
                              for ci in range(num_classes))
        print(f'Ep {ep:3d} | Acc={m["acc"]*100:.2f}% | mIoU={m["miou"]*100:.2f}% | {iou_str}')

        if m['miou'] > best_miou:
            best_miou = m['miou']
            torch.save(model.state_dict(), save_path)
            print(f'  => Best saved (mIoU={best_miou*100:.2f}%)')

    print(f'\n[{model_name}] Best mIoU = {best_miou*100:.2f}%')
    return history


def plot_history(history, model_name='Model'):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['test_loss'],  label='Test')
    axes[0].set_title('Loss'); axes[0].legend()

    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['test_acc'],  label='Test Acc')
    axes[1].set_title('Accuracy'); axes[1].legend()

    axes[2].plot(history['test_miou'], label='mIoU', linewidth=2, color='black')
    n_cls      = len(history['iou_per_class'])
    cls_labels = ['Pot+BG', 'Plant'] if n_cls == 2 else ['BG', 'Pot', 'Plant']
    cls_colors = ['#FF8C00', '#33CC33', '#4488FF'][:n_cls]
    for ci in range(n_cls):
        axes[2].plot(history['iou_per_class'][ci],
                     label=cls_labels[ci], linestyle='--',
                     color=cls_colors[ci])
    axes[2].set_title('IoU per class'); axes[2].legend(); axes[2].set_ylim(0, 1)

    fig.suptitle(f'{model_name} ({INPUT_MODE}, {IN_CH}D)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()






EPOCHS = 100



### PointNet++ (ưu tiên #1)


In [ ]:
model_pp = PointNetPPSeg(NUM_CLASSES, IN_CH).to(device)
hist_pp  = train_model(model_pp, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='pp_best.pth', model_name='PointNet++')
plot_history(hist_pp, 'PointNet++')





### KPConv (ưu tiên #2)


In [ ]:
model_kp = KPConvSeg(NUM_CLASSES, IN_CH).to(device)
hist_kp  = train_model(model_kp, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='kp_best.pth', model_name='KPConv')
plot_history(hist_kp, 'KPConv')




### DGCNN (ưu tiên #3)


In [ ]:
model_dg = DGCNNSeg(NUM_CLASSES, IN_CH, k=20).to(device)
hist_dg  = train_model(model_dg, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='dg_best.pth', model_name='DGCNN')
plot_history(hist_dg, 'DGCNN')





### PointNeXt (SOTA, thay PCT do OOM)


In [ ]:
model_nx = PointNeXtSeg(NUM_CLASSES, IN_CH).to(device)
hist_nx  = train_model(model_nx, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='pointnext_best.pth', model_name='PointNeXt')
plot_history(hist_nx, 'PointNeXt')





### PointNet baseline


In [ ]:
model_pn = PointNetSeg(NUM_CLASSES, IN_CH).to(device)
hist_pn  = train_model(model_pn, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='pn_best.pth', model_name='PointNet')
plot_history(hist_pn, 'PointNet')




### Point Transformer v1 (attention-based)


In [ ]:
# Lưu ý: nếu OOM, giảm BATCH_SIZE xuống 4 hoặc k=12
model_pt = PointTransformerSeg(NUM_CLASSES, IN_CH, k=16).to(device)
hist_pt  = train_model(model_pt, train_loader, test_loader,
                        NUM_CLASSES, class_weights, EPOCHS,
                        save_path='pt_best.pth', model_name='PointTrans')
plot_history(hist_pt, 'PointTransformer')






ablation = {}
for mode, dim in [('xyz', 3), ('xyz_h', 4), ('xyz_h_n', 7)]:
    print(f'\n{"="*50}  {mode} ({dim}D)')

    tr = RicePointCloudDataset(train_files, NUM_POINTS, mode, augment=True,  normal_cache=normal_cache)
    te = RicePointCloudDataset(test_files,  NUM_POINTS, mode, augment=False, normal_cache=normal_cache)
    tl = DataLoader(tr, BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2)
    el = DataLoader(te, BATCH_SIZE, shuffle=False,                  num_workers=2)

    m = PointNetPPSeg(NUM_CLASSES, dim).to(device)
    h = train_model(m, tl, el, NUM_CLASSES, class_weights,
                    epochs=50, save_path=f'pp_{mode}.pth', model_name=f'PP-{mode}')

    bi = int(np.argmax(h['test_miou']))
    row = {
        'in_ch':      dim,
        'best_miou':  f"{h['test_miou'][bi]*100:.2f}%",
        'best_acc':   f"{h['test_acc'][bi]*100:.2f}%",
    }
    # Lấy IoU theo num_classes thực tế
    n_cls  = len(h['iou_per_class'])
    labels = ['pot_bg', 'plant'] if n_cls == 2 else ['bg', 'pot', 'plant']
    for ci, lab in enumerate(labels):
        row[f'iou_{lab}'] = f"{h['iou_per_class'][ci][bi]*100:.2f}%"
    ablation[mode] = row

print('\n' + '='*55)
print('ABLATION — PointNet++ theo input mode')
print('='*55)
df_ab = pd.DataFrame(ablation).T
print(df_ab.to_string())





all_hist = {
    'PointNet':    hist_pn,
    'PointNet++':  hist_pp,
    'KPConv':      hist_kp,
    'DGCNN':       hist_dg,
    'PointNeXt':   hist_nx,
    'PointTrans':  hist_pt,
}

rows = {}
for name, h in all_hist.items():
    bi = int(np.argmax(h['test_miou']))
    row = {
        'Best Epoch':   bi + 1,
        'Test Acc (%)': round(h['test_acc'][bi]  * 100, 2),
        'mIoU (%)':     round(h['test_miou'][bi] * 100, 2),
    }
    if 'iou_per_class' in h:
        n_cls  = len(h['iou_per_class'])
        labels = ['Pot+BG', 'Plant'] if n_cls == 2 else ['BG', 'Pot', 'Plant']
        for ci, lab in enumerate(labels):
            row[f'IoU {lab} (%)'] = round(h['iou_per_class'][ci][bi] * 100, 2)
    rows[name] = row

df = pd.DataFrame(rows).T.sort_values('mIoU (%)', ascending=False)
print(f'Input: {INPUT_MODE} ({IN_CH}D)  |  NUM_CLASSES={NUM_CLASSES}  |  EPOCHS={EPOCHS}\n')
print(df.to_string())



### Bar chart


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
names  = df.index.tolist()
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#9467BD', '#8C564B']

ax = axes[0]
x  = np.arange(len(names))
iou_cols = [c for c in df.columns if c.startswith('IoU ')]
w  = 0.8 / max(len(iou_cols), 1)
offset = -(len(iou_cols) - 1) / 2 * w
for i, col in enumerate(iou_cols):
    vals  = df[col].tolist()
    label = col.replace('IoU ', '').replace(' (%)', '')
    ax.bar(x + (offset + i * w), vals, w, label=label)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15)
ax.set_title('IoU per class', fontsize=12); ax.legend()
ax.set_ylim(0, 105); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
miou_vals = df['mIoU (%)'].tolist()
bars = ax2.bar(names, miou_vals, color=colors[:len(names)], edgecolor='black', width=0.55)
for b, v in zip(bars, miou_vals):
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
             f'{v:.1f}%', ha='center', fontweight='bold')
ax2.set_title('Best mIoU (%)', fontsize=12); ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticklabels(names, rotation=15)

fig.suptitle(f'So sánh mô hình — {INPUT_MODE} ({IN_CH}D), {EPOCHS} epochs',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()



### Đường cong mIoU theo epoch


In [ ]:
plt.figure(figsize=(11, 5))
for name, h in all_hist.items():
    plt.plot(h['test_miou'], label=name, linewidth=1.8)
plt.xlabel('Epoch'); plt.ylabel('Test mIoU')
plt.title(f'Test mIoU theo Epoch — {EPOCHS} epochs')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('miou_curves.png', dpi=200, bbox_inches='tight')
plt.show()






# ════════════════════════════════════════════════
#  VISUALIZATION CHO BÁO CÁO — Setup
# ════════════════════════════════════════════════
import seaborn as sns
from sklearn.metrics import confusion_matrix
from matplotlib.patches import Patch

COLOR_MAP  = {0: '#FF8C00', 1: '#33CC33'} if NUM_CLASSES == 2 else \
             {0: '#4488FF', 1: '#FF8C00', 2: '#33CC33'}
CLASS_NAME = {0: 'Pot/Chậu', 1: 'Plant/Cây'} if NUM_CLASSES == 2 else \
             {0: 'Background', 1: 'Pot', 2: 'Plant'}

# Tự tìm best model
best_name = max(all_hist, key=lambda n: max(all_hist[n]['test_miou']))
print(f'Best model: {best_name}  '
      f'(mIoU={max(all_hist[best_name]["test_miou"])*100:.2f}%)')

def load_best(cls, path, **kw):
    m = cls(num_classes=NUM_CLASSES, in_channel=IN_CH, **kw).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval(); return m

best_models = {
    'PointNet':    load_best(PointNetSeg,         'pn_best.pth'),
    'PointNet++':  load_best(PointNetPPSeg,       'pp_best.pth'),
    'KPConv':      load_best(KPConvSeg,           'kp_best.pth'),
    'DGCNN':       load_best(DGCNNSeg,            'dg_best.pth', k=20),
    'PointNeXt':   load_best(PointNeXtSeg,        'pointnext_best.pth'),
    'PointTrans':  load_best(PointTransformerSeg, 'pt_best.pth', k=16),
}






def viz_gt_vs_pred(models_dict, dataset, indices, save_name='gt_vs_pred.png'):
    n_samples = len(indices)
    n_cols    = len(models_dict) + 1
    fig = plt.figure(figsize=(3.5*n_cols, 4*n_samples))

    def scatter3d(ax, pts, labels, title):
        c = [COLOR_MAP.get(l, '#888') for l in labels]
        ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=c, s=3, alpha=0.7)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_facecolor('#fafafa'); ax.axis('off')

    for row, idx in enumerate(indices):
        pts, gt = dataset[idx]
        pts_t   = pts.unsqueeze(0).to(device)
        xyz_np  = pts[:3, :].T.numpy()
        gt_np   = gt.numpy()

        ax = fig.add_subplot(n_samples, n_cols, row*n_cols + 1, projection='3d')
        scatter3d(ax, xyz_np, gt_np, f'Ground Truth\n(Sample #{idx})')

        for j, (name, mdl) in enumerate(models_dict.items()):
            with torch.no_grad():
                pred = mdl(pts_t).argmax(-1).squeeze(0).cpu().numpy()
            acc = (pred == gt_np).mean() * 100
            ious = []
            for c in range(NUM_CLASSES):
                tp = ((pred==c)&(gt_np==c)).sum()
                d  = ((pred==c)|(gt_np==c)).sum()
                ious.append(tp/d if d > 0 else 0)
            miou = np.mean(ious) * 100
            ax = fig.add_subplot(n_samples, n_cols, row*n_cols + j + 2, projection='3d')
            scatter3d(ax, xyz_np, pred, f'{name}\nAcc={acc:.1f}% mIoU={miou:.1f}%')

    fig.legend(handles=[Patch(color=COLOR_MAP[c], label=CLASS_NAME[c])
                         for c in range(NUM_CLASSES)],
               loc='lower center', ncol=NUM_CLASSES, fontsize=11,
               bbox_to_anchor=(0.5, -0.005))
    plt.suptitle('So sánh kết quả phân đoạn 3D — Geometry-Only Input',
                 fontsize=14, fontweight='bold', y=1.0)
    plt.tight_layout()
    plt.savefig(save_name, dpi=180, bbox_inches='tight')
    plt.show()

random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), min(3, len(test_dataset)))
viz_gt_vs_pred(best_models, test_dataset, sample_indices)






def plot_confusion(model, loader, model_name, save_name=None):
    model.eval()
    all_p = []; all_g = []
    with torch.no_grad():
        for pts, lbl in loader:
            pts = pts.to(device)
            pred = model(pts).argmax(-1).cpu().numpy().ravel()
            all_p.append(pred)
            all_g.append(lbl.numpy().ravel())
    P = np.concatenate(all_p)
    G = np.concatenate(all_g)
    cm = confusion_matrix(G, P, labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = [CLASS_NAME[c] for c in range(NUM_CLASSES)]

    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=axes[0])
    axes[0].set_title(f'{model_name} — Counts')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Greens',
                xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title(f'{model_name} — Normalized (per-row)')
    axes[1].set_xlabel('Predicted')

    plt.tight_layout()
    if save_name:
        plt.savefig(save_name, dpi=180, bbox_inches='tight')
    plt.show()
    return cm

cm = plot_confusion(best_models[best_name], test_loader, best_name,
                    save_name=f'confusion_{best_name}.png')

print(f'\n{best_name} — Per-class metrics:')
print(f'{"Class":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
for c in range(NUM_CLASSES):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    prec = tp / (tp+fp) if (tp+fp)>0 else 0
    rec  = tp / (tp+fn) if (tp+fn)>0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec)>0 else 0
    print(f'{CLASS_NAME[c]:<12} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {cm[c,:].sum():>10,}')






def viz_errors(model, dataset, idx, model_name, save_name=None):
    pts, gt = dataset[idx]
    pts_t   = pts.unsqueeze(0).to(device)
    xyz_np  = pts[:3, :].T.numpy()
    gt_np   = gt.numpy()
    with torch.no_grad():
        pred = model(pts_t).argmax(-1).squeeze(0).cpu().numpy()

    correct = pred == gt_np
    err_pot_to_plant = (gt_np == 0) & (pred == 1)
    err_plant_to_pot = (gt_np == 1) & (pred == 0)

    fig = plt.figure(figsize=(20, 5))

    ax = fig.add_subplot(1, 4, 1, projection='3d')
    c = [COLOR_MAP[l] for l in gt_np]
    ax.scatter(xyz_np[:,0], xyz_np[:,1], xyz_np[:,2], c=c, s=3, alpha=0.7)
    ax.set_title('Ground Truth', fontweight='bold'); ax.axis('off')

    ax = fig.add_subplot(1, 4, 2, projection='3d')
    c = [COLOR_MAP[l] for l in pred]
    ax.scatter(xyz_np[:,0], xyz_np[:,1], xyz_np[:,2], c=c, s=3, alpha=0.7)
    ax.set_title(f'Prediction ({model_name})', fontweight='bold'); ax.axis('off')

    ax = fig.add_subplot(1, 4, 3, projection='3d')
    cc = ['#22CC22' if c else '#DD2222' for c in correct]
    ax.scatter(xyz_np[:,0], xyz_np[:,1], xyz_np[:,2], c=cc, s=3, alpha=0.7)
    ax.set_title(f'Correct={correct.sum()/len(correct)*100:.1f}%  '
                 f'(xanh) vs Error (đỏ)', fontweight='bold'); ax.axis('off')

    ax = fig.add_subplot(1, 4, 4, projection='3d')
    cc = []
    for i in range(len(gt_np)):
        if err_pot_to_plant[i]: cc.append('#FF0000')
        elif err_plant_to_pot[i]: cc.append('#0000FF')
        else: cc.append('#DDDDDD')
    ax.scatter(xyz_np[:,0], xyz_np[:,1], xyz_np[:,2], c=cc, s=3, alpha=0.7)
    ax.set_title(f'Error types\n'
                 f'Pot→Plant: {err_pot_to_plant.sum()}  '
                 f'Plant→Pot: {err_plant_to_pot.sum()}', fontweight='bold')
    ax.axis('off')

    plt.suptitle(f'Error Analysis — {model_name} — Sample #{idx}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_name:
        plt.savefig(save_name, dpi=180, bbox_inches='tight')
    plt.show()

viz_errors(best_models[best_name], test_dataset, sample_indices[0], best_name,
           save_name=f'error_analysis_{best_name}.png')






# ════════════════════════════════════════════════


### PHASE 2 — Pipeline tính chiều cao cây thực


In [ ]:
#  Cải tiến:
#   - Percentile 99 thay max (lọc 1% điểm noise cao nhất)
#   - DBSCAN clustering: bỏ cluster nhỏ < 100 điểm (outliers bay lơ lửng)
#   - Export file _plant.ply và _pot.ply
# ════════════════════════════════════════════════
from sklearn.cluster import DBSCAN
from plyfile import PlyData, PlyElement

POT_REAL_HEIGHT_CM = 14.7    # Chiều cao chậu thực (ground truth scale)
PERCENTILE_POT     = 99.0    # Pot: dùng 99 (chậu đặc, ít noise cao)
PERCENTILE_PLANT   = 100    # Plant: dùng 99.9 (lá đỉnh thưa, cần lấy gần max)
MIN_CLUSTER_SIZE   = 100     # Cluster < 100 điểm = outliers, bỏ
DBSCAN_EPS         = 0.02    # Bán kính DBSCAN trên xyz đã normalize


def filter_outliers(xyz, eps=DBSCAN_EPS, min_size=MIN_CLUSTER_SIZE):
    """Loại outliers bằng DBSCAN — subsample nếu quá lớn để tránh OOM."""
    if len(xyz) < min_size:
        return np.ones(len(xyz), dtype=bool)
    
    # Subsample nếu > 50k điểm (DBSCAN O(n²) memory)
    MAX_PTS = 50000
    if len(xyz) > MAX_PTS:
        sub_idx = np.random.choice(len(xyz), MAX_PTS, replace=False)
        sub_xyz = xyz[sub_idx]
    else:
        sub_idx = np.arange(len(xyz))
        sub_xyz = xyz
    
    db = DBSCAN(eps=eps, min_samples=10).fit(sub_xyz)
    
    # Tìm clusters lớn
    big_clusters = set()
    for cl in set(db.labels_):
        if cl == -1: continue
        if (db.labels_ == cl).sum() >= min_size:
            big_clusters.add(cl)
    
    if not big_clusters:
        return np.ones(len(xyz), dtype=bool)
    
    # Lấy centroid các cluster lớn
    from scipy.spatial import cKDTree
    valid_sub = np.isin(db.labels_, list(big_clusters))
    tree = cKDTree(sub_xyz[valid_sub])
    
    # Map toàn bộ xyz: điểm nào gần cluster lớn (< eps*2) → valid
    dists, _ = tree.query(xyz, k=1)
    valid = dists < eps * 3
    return valid


def segment_full_pointcloud(model, file_path, num_points=4096, n_passes=3):
    """
    Phân đoạn TOÀN BỘ điểm trong file PLY.
    
    Chiến lược chunked inference:
    - Bước 1: Chia full cloud thành N chunks 4096 điểm (covering, không sót)
    - Bước 2: Vote thêm n_passes lần với random sampling (smoothing)
    - rel_h và normal được tính TRÊN FULL CLOUD trước, không tính trên chunk
    """
    data = read_ply(file_path)
    xyz_full = data['xyz']
    N_total  = xyz_full.shape[0]

    # ── Normalize XYZ trên full cloud ──
    centroid = xyz_full.mean(0)
    xyz_n    = xyz_full - centroid
    scale    = np.max(np.linalg.norm(xyz_n, axis=1))
    if scale > 0: xyz_n = xyz_n / scale

    # ── Tính rel_h và normal TRÊN FULL CLOUD (consistency) ──
    rel_h_full = (xyz_n[:, 2] - xyz_n[:, 2].min()).reshape(-1, 1)
    
    if INPUT_MODE == 'xyz_h_n':
        if file_path in normal_cache:
            normal_full = normal_cache[file_path]
        else:
            # Tính normal trên FULL cloud, không phải trên 4096 sample
            normal_full = compute_normals(xyz_full, k=20)

    def make_feat(indices):
        """Tạo feature tensor cho subset chỉ định, dùng full-cloud features."""
        xyz_s = xyz_n[indices]
        if INPUT_MODE == 'xyz':
            return xyz_s
        elif INPUT_MODE == 'xyz_h':
            return np.concatenate([xyz_s, rel_h_full[indices]], axis=1)
        elif INPUT_MODE == 'xyz_h_n':
            return np.concatenate([xyz_s, rel_h_full[indices],
                                    normal_full[indices]], axis=1)

    votes  = np.zeros((N_total, NUM_CLASSES), dtype=np.float32)
    counts = np.zeros(N_total, dtype=np.int32)

    model.eval()
    with torch.no_grad():
        # ── Bước 1: Chunked inference — đảm bảo MỌI điểm được vote ──
        # Permute toàn bộ index, chia thành chunks 4096
        all_idx = np.random.permutation(N_total)
        n_chunks = (N_total + num_points - 1) // num_points

        for ci in range(n_chunks):
            start = ci * num_points
            end   = min(start + num_points, N_total)
            chunk_idx = all_idx[start:end]
            
            # Pad nếu chunk cuối < num_points
            if len(chunk_idx) < num_points:
                pad_size = num_points - len(chunk_idx)
                pad_idx  = np.random.choice(chunk_idx, pad_size)
                chunk_full = np.concatenate([chunk_idx, pad_idx])
            else:
                chunk_full = chunk_idx

            feat = make_feat(chunk_full)
            x    = torch.from_numpy(feat).float().T.unsqueeze(0).to(device)
            out  = model(x).softmax(-1).squeeze(0).cpu().numpy()

            np.add.at(votes,  chunk_full, out)
            np.add.at(counts, chunk_full, 1)
            if ci % 20 == 0:
                torch.cuda.empty_cache()

        # ── Bước 2: Random passes để smoothing ──
        for _ in range(n_passes):
            choices = np.random.choice(N_total, num_points,
                                        replace=(N_total < num_points))
            feat = make_feat(choices)
            x    = torch.from_numpy(feat).float().T.unsqueeze(0).to(device)
            out  = model(x).softmax(-1).squeeze(0).cpu().numpy()
            np.add.at(votes,  choices, out)
            np.add.at(counts, choices, 1)

    # Mọi điểm bây giờ đều có ít nhất 1 vote (do chunked inference)
    not_voted = counts == 0
    if not_voted.any():
        # Fallback an toàn (gần như không xảy ra)
        votes[not_voted] = 1.0
    pred = votes.argmax(-1)
    torch.cuda.empty_cache()
    return xyz_full, pred


def compute_plant_height(xyz, label,
                         real_pot_height_cm=14.7,
                         pct_pot=PERCENTILE_POT,
                         pct_plant=PERCENTILE_PLANT,
                         use_dbscan=True):
    """Tính chiều cao cây thực dùng chậu làm tỉ lệ.
    
    Cải tiến:
    - DBSCAN loại cluster nhỏ (outliers bay lơ lửng)
    - Percentile riêng cho Pot và Plant:
      + Pot:   pct=99   (chậu đặc, ít noise → cắt 1% là đủ)
      + Plant: pct=99.9 (lá đỉnh thưa → cần lấy gần max để không cụt đỉnh)
    """
    pot_mask   = label == 0
    plant_mask = label == 1

    if pot_mask.sum() < 50 or plant_mask.sum() < 50:
        return None

    xyz_pot   = xyz[pot_mask]
    xyz_plant = xyz[plant_mask]

    # Lọc outlier bằng DBSCAN — đặc biệt quan trọng cho Plant 
    # vì DBSCAN sẽ loại các điểm bay lơ lửng cô lập trên đỉnh
    if use_dbscan:
        valid_pot   = filter_outliers(xyz_pot)
        valid_plant = filter_outliers(xyz_plant)
        if valid_pot.sum()   > 50: xyz_pot   = xyz_pot[valid_pot]
        if valid_plant.sum() > 50: xyz_plant = xyz_plant[valid_plant]

    # Đo theo percentile riêng cho từng class
    z_pot_top   = np.percentile(xyz_pot[:,   2], pct_pot)
    z_pot_bot   = np.percentile(xyz_pot[:,   2], 100 - pct_pot)
    z_plant_top = np.percentile(xyz_plant[:, 2], pct_plant)

    pot_height_pc   = z_pot_top - z_pot_bot
    plant_height_pc = z_plant_top - z_pot_top

    if pot_height_pc <= 0:
        return None

    # ── Sanity check: cây phải cao hơn miệng chậu ──
    # Nếu z_plant_top < z_pot_top → segmentation có lỗi
    # (model nhầm điểm cây thành chậu ở phần trên)
    # → Fallback: dùng percentile cao hơn để loại noise
    if plant_height_pc <= 0:
        # Thử percentile chặt hơn (95) cho Pot để loại noise cao mạnh hơn
        z_pot_top_strict = np.percentile(xyz_pot[:, 2], 95)
        plant_height_pc  = z_plant_top - z_pot_top_strict
        if plant_height_pc > 0:
            z_pot_top = z_pot_top_strict
        else:
            # Vẫn âm → segmentation hỏng nặng, đánh dấu invalid
            print(f'  ⚠ Sample bị lỗi segmentation: z_pot_top={z_pot_top:.3f} > '
                  f'z_plant_top={z_plant_top:.3f} (cả khi percentile=95)')
            return None

    scale = real_pot_height_cm / pot_height_pc
    plant_height_cm = plant_height_pc * scale

    return {
        'n_pot':              int(pot_mask.sum()),
        'n_plant':            int(plant_mask.sum()),
        'n_pot_valid':        int(len(xyz_pot)),
        'n_plant_valid':      int(len(xyz_plant)),
        'pot_height_pc':      float(pot_height_pc),
        'plant_height_pc':    float(plant_height_pc),
        'scale_cm_per_unit':  float(scale),
        'plant_height_cm':    float(plant_height_cm),
        'z_pot_top':          float(z_pot_top),
        'z_pot_bot':          float(z_pot_bot),
        'z_plant_top':        float(z_plant_top),
    }


def export_segments_ply(xyz, pred, original_path, output_dir='./phase2_outputs'):
    """Tách point cloud → file PLY: _pot.ply và _plant.ply"""
    os.makedirs(output_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(original_path))[0]
    paths = {}
    for cls, name in [(0, 'pot'), (1, 'plant')]:
        mask = pred == cls
        if mask.sum() == 0:
            continue
        pts = xyz[mask].astype(np.float32)
        vertex = np.array(
            [(p[0], p[1], p[2]) for p in pts],
            dtype=[('x','f4'), ('y','f4'), ('z','f4')]
        )
        el = PlyElement.describe(vertex, 'vertex')
        out_path = os.path.join(output_dir, f'{base}_{name}.ply')
        PlyData([el]).write(out_path)
        paths[name] = out_path
    return paths


# ════════════════════════════════════════════════
#  Test trên 1 file
# ════════════════════════════════════════════════
sample_file = test_files[0]
print(f'Test file: {os.path.basename(sample_file)}')
print(f'Best model: {best_name}\n')

xyz_full, pred = segment_full_pointcloud(best_models[best_name], sample_file, n_passes=5)
print(f'Total points: {len(xyz_full):,}')
print(f'Predicted Pot:   {(pred==0).sum():,} ({(pred==0).mean()*100:.1f}%)')
print(f'Predicted Plant: {(pred==1).sum():,} ({(pred==1).mean()*100:.1f}%)')

# Export ply
print('\n=== Export PLY files ===')
ply_paths = export_segments_ply(xyz_full, pred, sample_file)
for name, p in ply_paths.items():
    n = (pred == (0 if name=='pot' else 1)).sum()
    print(f'  {p}  ({n:,} points)')

# Tính chiều cao
result = compute_plant_height(xyz_full, pred, POT_REAL_HEIGHT_CM,
                              pct_pot=PERCENTILE_POT,
                              pct_plant=PERCENTILE_PLANT,
                              use_dbscan=True)
print(f'\n=== Phenotype Results ===')
print(f'Pot points (raw → valid)  : {result["n_pot"]:,} → {result["n_pot_valid"]:,}')
print(f'Plant points (raw → valid): {result["n_plant"]:,} → {result["n_plant_valid"]:,}')
print(f'z_pot_bot                 : {result["z_pot_bot"]:.4f}')
print(f'z_pot_top                 : {result["z_pot_top"]:.4f}')
print(f'z_plant_top               : {result["z_plant_top"]:.4f}')
print(f'Pot height   (PC)         : {result["pot_height_pc"]:.4f}')
print(f'Plant height (PC)         : {result["plant_height_pc"]:.4f}')
print(f'Scale                     : {result["scale_cm_per_unit"]:.4f} cm per unit_PC')
print(f'>>> Plant height (REAL)   : {result["plant_height_cm"]:.2f} cm')






def viz_height_measurement(xyz, pred, result=None, title='', save_name=None):
    fig = plt.figure(figsize=(16, 6))

    # 3D view
    ax = fig.add_subplot(1, 2, 1, projection='3d')
    c = [COLOR_MAP[l] for l in pred]
    ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=c, s=2, alpha=0.6)

    if result is not None:
        # Đường ngang ở 2 mức z khác nhau
        x_range = [xyz[:,0].min(), xyz[:,0].max()]
        y_range = [xyz[:,1].min(), xyz[:,1].max()]
        # Pot top (đỏ)
        ax.plot(x_range, [y_range[0]]*2, [result['z_pot_top']]*2,
                'r--', linewidth=2.5, label=f'Pot top z={result["z_pot_top"]:.3f}')
        # Plant top (xanh)
        ax.plot(x_range, [y_range[0]]*2, [result['z_plant_top']]*2,
                'g--', linewidth=2.5, label=f'Plant top z={result["z_plant_top"]:.3f}')
        ax.legend(fontsize=9, loc='upper left')
    ax.set_title('3D View — Predicted segmentation', fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

    # XZ side view (rõ nhất để thấy chiều cao)
    ax2 = fig.add_subplot(1, 2, 2)
    for c in range(NUM_CLASSES):
        m = pred == c
        ax2.scatter(xyz[m, 0], xyz[m, 2], c=COLOR_MAP[c], s=2, alpha=0.5,
                    label=CLASS_NAME[c])

    if result is not None:
        ax2.axhline(result['z_pot_top'],   color='red',   linestyle='--', linewidth=2,
                    label=f'Pot top  ({POT_REAL_HEIGHT_CM} cm reference)')
        ax2.axhline(result['z_plant_top'], color='green', linestyle='--', linewidth=2,
                    label=f'Plant top')
        x_mid = (xyz[:,0].min() + xyz[:,0].max()) / 2
        ax2.annotate('', xy=(x_mid, result['z_plant_top']),
                     xytext=(x_mid, result['z_pot_top']),
                     arrowprops=dict(arrowstyle='<->', color='black', lw=2))
        ax2.text(x_mid + 0.05, (result['z_plant_top']+result['z_pot_top'])/2,
                 f'  {result["plant_height_cm"]:.2f} cm',
                 fontsize=14, fontweight='bold',
                 bbox=dict(facecolor='yellow', alpha=0.8, edgecolor='black'))

    ax2.set_xlabel('X'); ax2.set_ylabel('Z (height)')
    ax2.set_title('Side View (XZ) — Height measurement', fontweight='bold')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(alpha=0.3)
    ax2.set_aspect('equal')

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_name:
        plt.savefig(save_name, dpi=180, bbox_inches='tight')
    plt.show()


viz_height_measurement(xyz_full, pred, result=result,
                       title=f'{os.path.basename(sample_file)} — '
                             f'Plant Height = {result["plant_height_cm"]:.2f} cm '
                             f'(model: {best_name})',
                       save_name='phase2_height_measurement.png')






# Đo chiều cao + export PLY cho cả test set
print(f'Đo chiều cao + export ply cho {len(test_files)} file test...\n')

results_phase2 = []
output_dir = './phase2_outputs'
os.makedirs(output_dir, exist_ok=True)

for fp in tqdm(test_files, desc='Processing'):
    try:
        xyz_full, pred = segment_full_pointcloud(
            best_models[best_name], fp, n_passes=3)

        # Export ply
        export_segments_ply(xyz_full, pred, fp, output_dir=output_dir)

        # Tính chiều cao
        r = compute_plant_height(xyz_full, pred, POT_REAL_HEIGHT_CM,
                                 pct_pot=PERCENTILE_POT,
                                 pct_plant=PERCENTILE_PLANT,
                                 use_dbscan=True)
        if r is not None:
            r['file'] = os.path.basename(fp)
            r['n_total'] = len(xyz_full)
            results_phase2.append(r)
    except Exception as e:
        print(f'  Error {fp}: {e}')

df_h = pd.DataFrame(results_phase2)
df_h = df_h[['file', 'n_total', 'n_pot', 'n_plant',
             'n_pot_valid', 'n_plant_valid',
             'pot_height_pc', 'plant_height_pc',
             'scale_cm_per_unit', 'plant_height_cm']]
print(df_h.head(10).to_string(index=False))

print(f'\n=== Phase 2 — Plant Height Statistics ===')
print(f'Số mẫu thành công     : {len(df_h)} / {len(test_files)}')
print(f'Plant height (mean)   : {df_h["plant_height_cm"].mean():.2f} cm')
print(f'Plant height (std)    : {df_h["plant_height_cm"].std():.2f} cm')
print(f'Plant height (min)    : {df_h["plant_height_cm"].min():.2f} cm')
print(f'Plant height (max)    : {df_h["plant_height_cm"].max():.2f} cm')
print(f'Plant height (median) : {df_h["plant_height_cm"].median():.2f} cm')

df_h.to_csv('phase2_heights.csv', index=False)
print(f'\nĐã lưu:')
print(f'  - phase2_heights.csv  (số liệu)')
print(f'  - {output_dir}/  (file ply: <name>_pot.ply, <name>_plant.ply)')





### Histogram phân bố chiều cao


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df_h['plant_height_cm'], bins=20, color='#33CC33',
             edgecolor='black', alpha=0.8)
axes[0].axvline(df_h['plant_height_cm'].mean(), color='red',
                linestyle='--', linewidth=2,
                label=f'Mean = {df_h["plant_height_cm"].mean():.2f} cm')
axes[0].axvline(df_h['plant_height_cm'].median(), color='blue',
                linestyle='--', linewidth=2,
                label=f'Median = {df_h["plant_height_cm"].median():.2f} cm')
axes[0].set_xlabel('Plant Height (cm)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Phân bố chiều cao cây — Test set')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].boxplot([df_h['plant_height_cm']], labels=['Plant Height'],
                showmeans=True, patch_artist=True,
                boxprops=dict(facecolor='#33CC33', alpha=0.6),
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
axes[1].set_ylabel('Height (cm)')
axes[1].set_title('Boxplot — Plant Height Distribution')
axes[1].grid(alpha=0.3)

plt.suptitle(f'Phase 2 Results — {best_name} — Pot reference = {POT_REAL_HEIGHT_CM} cm',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2_height_distribution.png', dpi=180, bbox_inches='tight')
plt.show()






get_ipython().getoutput("pip install dbscan")


# ════════════════════════════════════════════════


### PHASE 2B — Đếm nhánh (v1: multi-slice DBSCAN + max voting)


In [ ]:
#  
#  Phương pháp tốt nhất qua nhiều thử nghiệm:
#    - MAE = 3.33, Correlation = 0.86, Slope = 0.48
#    - Tốt hơn baseline "đoán mean" (MAE 3.61)
#    - Có nguyên lý vật lý: đếm tại vùng cây nhánh tách rõ
# ════════════════════════════════════════════════
from scipy.spatial.distance import pdist
from sklearn.cluster import DBSCAN

# Tham số
SLICE_THICKNESS  = 0.01      # 1cm
SLICE_OFFSETS    = [0.05, 0.07, 0.09, 0.11, 0.13]  # 5-13cm trên miệng chậu
DBSCAN_EPS_TILL  = 0.008     # 8mm — khoảng cách giữa 2 thân khác nhau
DBSCAN_MIN_TILL  = 3
MAX_STEM_DIAM    = 0.030     # cluster > 3cm là lá, không phải thân
POT_SHRINK       = 1.0       # đếm cả vùng chậu


def read_ply_xyz(file_path):
    v = PlyData.read(file_path)['vertex']
    return np.vstack([v['x'], v['y'], v['z']]).T.astype(np.float32)


def get_pot_top_and_radius(pot_xyz, percentile_top=99.0):
    """Xác định miệng chậu + tâm + bán kính."""
    z_top = np.percentile(pot_xyz[:, 2], percentile_top)
    rim_mask = (pot_xyz[:, 2] >= z_top - 0.02) & (pot_xyz[:, 2] <= z_top + 0.005)
    rim_xy   = pot_xyz[rim_mask, :2]
    if len(rim_xy) < 10:
        mask = pot_xyz[:, 2] >= np.percentile(pot_xyz[:, 2], 90)
        rim_xy = pot_xyz[mask, :2]
    center_xy = rim_xy.mean(axis=0)
    dists = np.linalg.norm(rim_xy - center_xy, axis=1)
    radius = np.percentile(dists, 95)
    return z_top, center_xy, radius


def count_tillers_one_slice(plant_xyz, z_low, z_high, center_xy, radius,
                             eps=DBSCAN_EPS_TILL, min_pts=DBSCAN_MIN_TILL,
                             max_diam=MAX_STEM_DIAM):
    """Đếm cluster (= nhánh) trong 1 lát cắt giới hạn vùng chậu."""
    z_mask = (plant_xyz[:, 2] >= z_low) & (plant_xyz[:, 2] < z_high)
    if z_mask.sum() < min_pts:
        return 0, None, None
    
    slice_pts = plant_xyz[z_mask]
    dists = np.linalg.norm(slice_pts[:, :2] - center_xy, axis=1)
    in_pot = slice_pts[dists <= radius * POT_SHRINK]
    if len(in_pot) < min_pts:
        return 0, None, None
    
    xy = in_pot[:, :2]
    labels = DBSCAN(eps=eps, min_samples=min_pts).fit(xy).labels_
    
    n_stems = 0; valid_pts = []; valid_lbls = []
    for lbl in set(labels) - {-1}:
        cluster_pts = xy[labels == lbl]
        if len(cluster_pts) < min_pts: continue
        if len(cluster_pts) > 1 and pdist(cluster_pts).max() > max_diam:
            continue   # cluster to → là lá
        n_stems += 1
        valid_pts.append(cluster_pts)
        valid_lbls.extend([lbl] * len(cluster_pts))
    
    if valid_pts:
        return n_stems, np.vstack(valid_pts), np.array(valid_lbls)
    return n_stems, None, None


def count_tillers(plant_path, pot_path, verbose=False):
    """Pipeline chính: multi-slice voting."""
    plant_xyz = read_ply_xyz(plant_path)
    pot_xyz   = read_ply_xyz(pot_path)
    z_pot_top, center_xy, radius = get_pot_top_and_radius(pot_xyz)
    
    counts, slice_data = [], []
    for offset in SLICE_OFFSETS:
        z_low  = z_pot_top + offset
        z_high = z_low + SLICE_THICKNESS
        n, pts, lbls = count_tillers_one_slice(
            plant_xyz, z_low, z_high, center_xy, radius)
        counts.append(n)
        slice_data.append({'offset_cm': offset*100, 'z_range': (z_low, z_high),
                           'n_stems': n, 'pts': pts, 'lbls': lbls})
        if verbose:
            print(f'  Offset {offset*100:5.1f}cm → {n} nhánh')
    
    # Voting: MAX (lấy lát thấy đủ nhánh nhất)
    n_final = int(np.max(counts))
    
    return {
        'n_tillers':     n_final,
        'voting_method': 'max',
        'mean_count':    float(np.mean(counts)),
        'all_counts':    counts,
        'pot_top_z':     z_pot_top,
        'pot_center_xy': center_xy,
        'pot_radius':    radius,
        'slice_data':    slice_data,
    }




### Test trên 1 cặp file


In [ ]:
sample_base = "C1.1_26410"  # đổi theo file của bạn
plant_p = f'/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs/{sample_base}_plant.ply'
pot_p   = f'/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs/{sample_base}_pot.ply'

print(f'Test: {sample_base}')
result_t = count_tillers(plant_p, pot_p, verbose=True)
print(f'\nCounts qua các lát: {result_t["all_counts"]}')
print(f'Voting method:      {result_t["voting_method"]}')
print(f'>>> Số nhánh:       {result_t["n_tillers"]}')


def visualize_tillers(result, plant_path, save_path=None):
    """Vẽ 5 lát cắt + bar chart counts."""
    plant_xyz = read_ply_xyz(plant_path)
    n_slices = len(result['slice_data'])
    
    fig = plt.figure(figsize=(20, 5))
    
    for i, sd in enumerate(result['slice_data']):
        ax = fig.add_subplot(1, n_slices + 1, i + 1)
        
        # Background: tất cả điểm Plant trong lát cắt (xám)
        z_low, z_high = sd['z_range']
        z_mask = (plant_xyz[:, 2] >= z_low) & (plant_xyz[:, 2] < z_high)
        bg = plant_xyz[z_mask, :2]
        if len(bg) > 0:
            ax.scatter(bg[:,0], bg[:,1], c='#DDDDDD', s=8, alpha=0.5)
        
        # Foreground: stem clusters (màu)
        if sd['pts'] is not None:
            ax.scatter(sd['pts'][:,0], sd['pts'][:,1], c=sd['lbls'],
                       cmap='tab20', s=20, alpha=0.9)
        
        # Vẽ vòng tròn chậu
        theta = np.linspace(0, 2*np.pi, 100)
        cx, cy = result['pot_center_xy']
        r = result['pot_radius'] * POT_SHRINK
        ax.plot(cx + r*np.cos(theta), cy + r*np.sin(theta),
                'r--', linewidth=1.5, label=f'pot rim r={r:.3f}')
        ax.plot(cx, cy, 'r+', markersize=12)
        
        ax.set_title(f"Offset {sd['offset_cm']:.1f}cm\n{sd['n_stems']} nhánh",
                     fontweight='bold')
        ax.set_xlabel('X'); ax.set_ylabel('Y')
        ax.set_aspect('equal'); ax.grid(alpha=0.3)
        if i == 0: ax.legend(fontsize=8)
    
    # Bar chart counts
    ax = fig.add_subplot(1, n_slices + 1, n_slices + 1)
    offsets = [sd['offset_cm'] for sd in result['slice_data']]
    counts  = [sd['n_stems']    for sd in result['slice_data']]
    ax.bar(range(len(counts)), counts, color='#33CC33', edgecolor='black')
    ax.axhline(result['n_tillers'], color='red', linestyle='--',
               label=f'Max = {result["n_tillers"]}')
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([f'{o:.1f}' for o in offsets])
    ax.set_xlabel('Offset (cm)')
    ax.set_ylabel('Số nhánh')
    ax.set_title(f'Voting → {result["n_tillers"]} nhánh', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    
    plt.suptitle(f'Tiller counting (v1: multi-slice max voting) — {os.path.basename(plant_path)}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


visualize_tillers(result_t, plant_p, save_path='phase2b_tiller_counting.png')


# Đếm nhánh cho TOÀN BỘ file trong phase2_outputs/
import glob

OUTPUT_DIR = '/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs'

plant_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_plant.ply'))
print(f'Tìm thấy {len(plant_files)} cặp file plant/pot...\n')

tiller_results = []
for plant_p in tqdm(plant_files, desc='Counting tillers'):
    base  = os.path.basename(plant_p).replace('_plant.ply', '')
    pot_p = f'{OUTPUT_DIR}/{base}_pot.ply'
    
    if not os.path.exists(pot_p):
        print(f'  Skip {base}: không có _pot.ply')
        continue
    
    try:
        r = count_tillers(plant_p, pot_p, verbose=False)
        tiller_results.append({
            'file': base,
            'n_tillers':  r['n_tillers'],
            'mean_count': round(r['mean_count'], 1),
            'counts':     str(r['all_counts']),
            'pot_radius': round(r['pot_radius'], 4),
        })
    except Exception as e:
        print(f'  Error {base}: {e}')

df_t = pd.DataFrame(tiller_results)
print(df_t.head(15).to_string(index=False))

print(f'\n=== Tiller Statistics ===')
print(f'Số mẫu thành công  : {len(df_t)} / {len(plant_files)}')
print(f'Tillers (mean)     : {df_t["n_tillers"].mean():.2f}')
print(f'Tillers (median)   : {df_t["n_tillers"].median():.0f}')
print(f'Tillers (min, max) : {df_t["n_tillers"].min()}, {df_t["n_tillers"].max()}')

df_t.to_csv('phase2b_tillers.csv', index=False)
print('\nĐã lưu: phase2b_tillers.csv')

# Histogram + boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(df_t['n_tillers'], bins=range(0, df_t['n_tillers'].max()+2),
        color='#33CC33', edgecolor='black', alpha=0.8)
ax.axvline(df_t['n_tillers'].mean(), color='red', linestyle='--',
           linewidth=2, label=f'Mean = {df_t["n_tillers"].mean():.1f}')
ax.set_xlabel('Số nhánh'); ax.set_ylabel('Frequency')
ax.set_title('Phân bố số nhánh — Test set')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.boxplot([df_t['n_tillers']], labels=['Tillers'],
           showmeans=True, patch_artist=True,
           boxprops=dict(facecolor='#33CC33', alpha=0.6))
ax.set_ylabel('Số nhánh')
ax.set_title('Boxplot — Tiller Distribution')
ax.grid(alpha=0.3)

plt.suptitle('Phase 2B — Tiller Counting (v1: multi-slice max voting)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2b_tiller_distribution.png', dpi=150, bbox_inches='tight')
plt.show()





# ════════════════════════════════════════════════


### PHASE 2B — Đếm nhánh: Canopy Height Map + Local Maxima


In [ ]:
#  
#  Nguyên lý: mỗi nhánh lúa tạo 1 peak (đỉnh cục bộ) trên
#  bản đồ chiều cao. Đếm peaks = đếm nhánh.
#  
#  Ưu điểm:
#    - Không phụ thuộc DBSCAN eps (bottleneck của v1)
#    - Ngọn nhánh tách biệt dù gốc dính nhau
#    - Lá rủ có z thấp → tự động bị loại bởi height threshold
#    - Code đơn giản, chạy nhanh (<0.1s/file)
# ════════════════════════════════════════════════
from scipy.ndimage import maximum_filter, gaussian_filter

# Tham số
VOXEL_SIZE       = 0.008     # 8mm grid
GAUSS_SIGMA      = 1.0       # Gaussian smoothing (pixels)
MIN_HEIGHT_RATIO = 0.25       # Peak phải >= 30% chiều cao cây mới đếm
PEAK_NEIGHBORHOOD = 3        # Footprint cho local maxima (pixels)


def read_ply_xyz(file_path):
    v = PlyData.read(file_path)['vertex']
    return np.vstack([v['x'], v['y'], v['z']]).T.astype(np.float32)


def count_tillers_heightmap(plant_xyz, voxel_size=VOXEL_SIZE,
                             sigma=GAUSS_SIGMA,
                             min_height_ratio=MIN_HEIGHT_RATIO):
    """
    Đếm nhánh bằng Canopy Height Map + Local Maxima Detection.
    
    Bước 1: Project point cloud lên grid XY
    Bước 2: Mỗi cell = z_max (chiều cao cao nhất)
    Bước 3: Gaussian smooth để giảm noise
    Bước 4: Detect local maxima = peaks = nhánh
    Bước 5: Filter peaks theo height threshold
    """
    x, y, z = plant_xyz[:,0], plant_xyz[:,1], plant_xyz[:,2]
    
    # Grid
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    nx = int((x_max - x_min) / voxel_size) + 1
    ny = int((y_max - y_min) / voxel_size) + 1
    
    # Height map: mỗi cell = z_max
    hmap = np.full((ny, nx), np.nan)
    xi = np.clip(((x - x_min) / voxel_size).astype(int), 0, nx-1)
    yi = np.clip(((y - y_min) / voxel_size).astype(int), 0, ny-1)
    
    for i in range(len(z)):
        if np.isnan(hmap[yi[i], xi[i]]) or z[i] > hmap[yi[i], xi[i]]:
            hmap[yi[i], xi[i]] = z[i]
    
    # Fill NaN
    hmap_filled = np.where(np.isnan(hmap), np.nanmin(hmap), hmap)
    
    # Smooth
    hmap_smooth = gaussian_filter(hmap_filled, sigma=sigma)
    
    # Local maxima
    footprint_size = max(3, PEAK_NEIGHBORHOOD)
    local_max = maximum_filter(hmap_smooth, size=footprint_size)
    peaks = (hmap_smooth == local_max)
    
    # Height threshold
    z_range = np.nanmax(hmap) - np.nanmin(hmap)
    z_threshold = np.nanmin(hmap) + z_range * min_height_ratio
    peaks = peaks & (hmap_smooth > z_threshold) & (~np.isnan(hmap))
    
    peak_coords = np.argwhere(peaks)
    peak_heights = hmap_smooth[peaks]
    
    return len(peak_coords), hmap, hmap_smooth, peaks, peak_coords, peak_heights


def count_tillers(plant_path, pot_path=None, verbose=False):
    """Pipeline chính."""
    plant_xyz = read_ply_xyz(plant_path)
    
    n_tillers, hmap, hmap_smooth, peaks, peak_coords, peak_heights = \
        count_tillers_heightmap(plant_xyz)
    
    if verbose:
        print(f'  Height map size: {hmap.shape}')
        print(f'  Z range: [{np.nanmin(hmap):.3f}, {np.nanmax(hmap):.3f}]')
        print(f'  Peaks found: {n_tillers}')
    
    return {
        'n_tillers':     n_tillers,
        'hmap':          hmap,
        'hmap_smooth':   hmap_smooth,
        'peaks':         peaks,
        'peak_coords':   peak_coords,
        'peak_heights':  peak_heights,
    }




### Test


In [ ]:
sample_base = "C1.1_26410"
plant_p = f'/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs/{sample_base}_plant.ply'

print(f'Test: {sample_base}')
result_t = count_tillers(plant_p, verbose=True)
print(f'>>> Số nhánh: {result_t["n_tillers"]}')


def visualize_tillers(result, plant_path, save_path=None):
    """Vẽ height map + peaks."""
    plant_xyz = read_ply_xyz(plant_path)
    hmap = result['hmap_smooth']
    peaks = result['peaks']
    peak_coords = result['peak_coords']
    n = result['n_tillers']
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Height map
    ax = axes[0]
    im = ax.imshow(hmap, cmap='viridis', origin='lower')
    ax.set_title('Canopy Height Map (smoothed)', fontweight='bold')
    plt.colorbar(im, ax=ax, label='Z (height)')
    
    # 2. Height map + peaks
    ax = axes[1]
    ax.imshow(hmap, cmap='viridis', origin='lower')
    if len(peak_coords) > 0:
        ax.scatter(peak_coords[:, 1], peak_coords[:, 0],
                   c='red', s=80, marker='*', linewidths=1,
                   edgecolors='white', label=f'{n} peaks')
    ax.set_title(f'Detected Peaks → {n} nhánh', fontweight='bold')
    ax.legend(fontsize=10)
    
    # 3. 3D view với peaks đánh dấu
    ax = fig.add_subplot(1, 3, 3, projection='3d')
    # Subsample plant cho hiển thị
    idx = np.random.choice(len(plant_xyz), min(8000, len(plant_xyz)), replace=False)
    ax.scatter(plant_xyz[idx,0], plant_xyz[idx,1], plant_xyz[idx,2],
               c='#33CC33', s=1, alpha=0.3)
    
    # Đánh dấu vị trí peaks trên 3D
    if len(peak_coords) > 0:
        x_min = plant_xyz[:,0].min()
        y_min = plant_xyz[:,1].min()
        peak_x = peak_coords[:, 1] * VOXEL_SIZE + x_min
        peak_y = peak_coords[:, 0] * VOXEL_SIZE + y_min
        peak_z = result['peak_heights']
        ax.scatter(peak_x, peak_y, peak_z,
                   c='red', s=100, marker='*', linewidths=1,
                   edgecolors='black', label=f'{n} peaks')
    ax.set_title('3D View + Peak locations', fontweight='bold')
    ax.legend()
    
    plt.suptitle(f'Tiller counting (Height Map) — {os.path.basename(plant_path)}'
                 f'  →  {n} nhánh',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


visualize_tillers(result_t, plant_p, save_path='phase2b_tiller_counting.png')


# Đếm nhánh cho TOÀN BỘ file
import glob

OUTPUT_DIR = '/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs'
plant_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_plant.ply'))
print(f'Tìm thấy {len(plant_files)} file plant...\n')

tiller_results = []
for plant_p in tqdm(plant_files, desc='Counting tillers'):
    base = os.path.basename(plant_p).replace('_plant.ply', '')
    try:
        r = count_tillers(plant_p, verbose=False)
        tiller_results.append({
            'file': base,
            'n_tillers': r['n_tillers'],
        })
    except Exception as e:
        print(f'  Error {base}: {e}')

df_t = pd.DataFrame(tiller_results)
print(df_t.to_string(index=False))

print(f'\n=== Tiller Statistics ===')
print(f'Số mẫu      : {len(df_t)} / {len(plant_files)}')
print(f'Mean         : {df_t["n_tillers"].mean():.2f}')
print(f'Median       : {df_t["n_tillers"].median():.0f}')
print(f'Min, Max     : {df_t["n_tillers"].min()}, {df_t["n_tillers"].max()}')

df_t.to_csv('phase2b_tillers.csv', index=False)
print('\nĐã lưu: phase2b_tillers.csv')

# Histogram
plt.figure(figsize=(10, 4))
plt.hist(df_t['n_tillers'], bins=range(0, df_t['n_tillers'].max()+2),
         color='#33CC33', edgecolor='black', alpha=0.8)
plt.axvline(df_t['n_tillers'].mean(), color='red', linestyle='--',
            linewidth=2, label=f'Mean = {df_t["n_tillers"].mean():.1f}')
plt.xlabel('Số nhánh'); plt.ylabel('Frequency')
plt.title('Phân bố số nhánh (Height Map method)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('phase2b_tiller_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


# ════════════════════════════════════════════════


### PHASE 2B — Visualization kết quả đếm nhánh


In [ ]:
# ════════════════════════════════════════════════

# Ground truth
GT_TILLERS = {
    'C1.1_26410':6, 'C1.1_26427':9, 'C1.2_26410':8, 'C1.2_26424':10,
    'C1.3_26410':6, 'C1.4_26410':11,'C1.4_26422':6, 'C1.4_26429':6,
    'C2.1_26416':12,'C2.1_26417':15,'C2.1_26423':19,'C2.2_26413':13,
    'C2.3_26413':19,'C2.3_26423':14,'C2.4_26410':13,'C2.4_26425':11,
    'C2.4_26427':11,'C2.4_26428':11,'C2.5_26410':11,'C2.5_26429':14,
    'C3.1_26416':12,'C3.1_26422':14,'C3.1_26423':14,'C3.1_26428':13,
    'C3.2_26413':20,'C3.2_26416':19,'C3.3_26410':23,'C3.3_26423':23,
    'C3.3_26425':23,'C3.4_26416':22,'C3.4_26428':21,
}

# Match GT với predictions
files_gt = [f for f in df_t['file'] if f in GT_TILLERS]
gt   = np.array([GT_TILLERS[f] for f in files_gt])
pred = np.array([df_t[df_t['file']==f]['n_tillers'].iloc[0] for f in files_gt])
err  = pred - gt

mae  = np.abs(err).mean()
corr = np.corrcoef(gt, pred)[0,1]
slope, intercept = np.polyfit(gt, pred, 1)



### Print bảng so sánh


In [ ]:
print(f'{"file":<12} {"GT":>4} {"Pred":>5} {"Err":>5}')
print('-'*32)
for i, f in enumerate(files_gt):
    print(f'{f:<12} {gt[i]:>4} {pred[i]:>5} {err[i]:>+5}')
print(f'\nMAE = {mae:.2f} | Correlation = {corr:.3f} | Slope = {slope:.2f}')

# ════════════════════════════════════════════════
#  Figure 1: Scatter GT vs Pred + Error bar chart
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Scatter GT vs Pred
ax = axes[0]
ax.scatter(gt, pred, c='#33CC33', s=80, edgecolors='black', linewidth=0.8, zorder=3)
max_v = max(gt.max(), pred.max()) + 2
ax.plot([0, max_v], [0, max_v], 'k--', alpha=0.5, label='y=x (perfect)')
ax.plot([0, max_v], [slope*0+intercept, slope*max_v+intercept],
        'r-', alpha=0.7, label=f'Fit: y={slope:.2f}x+{intercept:.1f}')
ax.set_xlabel('Ground Truth', fontsize=11)
ax.set_ylabel('Predicted', fontsize=11)
ax.set_title(f'GT vs Predicted\nMAE={mae:.2f}, r={corr:.3f}', fontsize=12, fontweight='bold')
ax.set_xlim(0, max_v); ax.set_ylim(0, max_v)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_aspect('equal')

# Panel 2: Error per file (bar chart)
ax = axes[1]
x = np.arange(len(files_gt))
colors_bar = ['#33CC33' if abs(e) <= 2 else '#FF8C00' if abs(e) <= 4 else '#DD2222' for e in err]
ax.bar(x, err, color=colors_bar, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(+2, color='gray', linestyle=':', alpha=0.5)
ax.axhline(-2, color='gray', linestyle=':', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(files_gt, rotation=90, fontsize=7)
ax.set_ylabel('Error (Pred - GT)')
ax.set_title('Error per sample\n(xanh: |err|≤2, cam: ≤4, đỏ: >4)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Panel 3: Error distribution histogram
ax = axes[2]
ax.hist(err, bins=range(int(err.min())-1, int(err.max())+2),
        color='#4488FF', edgecolor='black', alpha=0.8)
ax.axvline(0, color='red', linewidth=2, label='Perfect (err=0)')
ax.axvline(err.mean(), color='orange', linewidth=2, linestyle='--',
           label=f'Mean err = {err.mean():+.1f}')
ax.set_xlabel('Error (Pred - GT)')
ax.set_ylabel('Frequency')
ax.set_title('Error Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(f'Phase 2B — Tiller Counting: Height Map Method (n={len(files_gt)} samples)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2b_gt_vs_pred.png', dpi=200, bbox_inches='tight')
plt.show()

# ════════════════════════════════════════════════
#  Figure 2: Error theo nhóm GT + metrics bảng
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Boxplot error theo nhóm
ax = axes[0]
groups = {'GT ≤ 8': (0, 9), 'GT 9-13': (9, 14), 'GT 14-19': (14, 20), 'GT ≥ 20': (20, 30)}
group_data = []
group_labels = []
for label, (lo, hi) in groups.items():
    mask = (gt >= lo) & (gt < hi)
    if mask.sum() > 0:
        group_data.append(err[mask])
        group_labels.append(f'{label}\n(n={mask.sum()})')

bp = ax.boxplot(group_data, labels=group_labels, showmeans=True, patch_artist=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=6))
palette = ['#33CC33', '#4488FF', '#FF8C00', '#DD2222']
for patch, color in zip(bp['boxes'], palette[:len(group_data)]):
    patch.set_facecolor(color); patch.set_alpha(0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Error (Pred - GT)')
ax.set_title('Error distribution per GT group', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Panel 2: Metrics table
ax = axes[1]
ax.axis('off')
table_data = []
for label, (lo, hi) in groups.items():
    mask = (gt >= lo) & (gt < hi)
    if mask.sum() > 0:
        m_gt   = gt[mask].mean()
        m_pred = pred[mask].mean()
        m_mae  = np.abs(err[mask]).mean()
        m_corr = np.corrcoef(gt[mask], pred[mask])[0,1] if mask.sum() > 2 else float('nan')
        table_data.append([label, mask.sum(), f'{m_gt:.1f}', f'{m_pred:.1f}',
                          f'{m_mae:.2f}', f'{m_corr:.2f}' if not np.isnan(m_corr) else 'N/A'])

# Thêm row tổng
table_data.append(['ALL', len(gt), f'{gt.mean():.1f}', f'{pred.mean():.1f}',
                   f'{mae:.2f}', f'{corr:.3f}'])

table = ax.table(cellText=table_data,
                 colLabels=['Group', 'n', 'GT mean', 'Pred mean', 'MAE', 'Corr'],
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
# Bold last row
for j in range(6):
    table[len(table_data), j].set_text_props(fontweight='bold')
ax.set_title('Performance metrics per group', fontsize=12, fontweight='bold', pad=20)

plt.suptitle(f'Phase 2B — Analysis by plant size',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2b_error_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

# ════════════════════════════════════════════════
#  Figure 3: Height map visualization cho 1 sample
# ════════════════════════════════════════════════
sample_base = "C1.1_26410"
plant_p = f'/kaggle/input/datasets/haquanghung/rice-segmentation-result/phase2_outputs/{sample_base}_plant.ply'
plant_xyz = read_ply_xyz(plant_p)

_, hmap, hmap_smooth, peaks_mask, peak_coords, peak_heights = \
    count_tillers_heightmap(plant_xyz)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw height map
ax = axes[0]
im = ax.imshow(np.where(np.isnan(hmap), np.nanmin(hmap), hmap),
               cmap='viridis', origin='lower')
ax.set_title('Raw Height Map', fontweight='bold')
plt.colorbar(im, ax=ax, label='Z', shrink=0.8)

# Smoothed + peaks
ax = axes[1]
ax.imshow(hmap_smooth, cmap='viridis', origin='lower')
if len(peak_coords) > 0:
    ax.scatter(peak_coords[:,1], peak_coords[:,0],
               c='red', s=120, marker='*', linewidths=1,
               edgecolors='white', zorder=5,
               label=f'{len(peak_coords)} peaks')
ax.set_title(f'Smoothed + Peaks → {len(peak_coords)} nhánh', fontweight='bold')
ax.legend(fontsize=10)

# 3D view + peaks
ax = fig.add_subplot(1, 3, 3, projection='3d')
idx_sub = np.random.choice(len(plant_xyz), min(8000, len(plant_xyz)), replace=False)
ax.scatter(plant_xyz[idx_sub,0], plant_xyz[idx_sub,1], plant_xyz[idx_sub,2],
           c='#33CC33', s=1, alpha=0.3)
if len(peak_coords) > 0:
    px = peak_coords[:,1] * VOXEL_SIZE + plant_xyz[:,0].min()
    py = peak_coords[:,0] * VOXEL_SIZE + plant_xyz[:,1].min()
    pz = peak_heights
    ax.scatter(px, py, pz, c='red', s=150, marker='*',
               edgecolors='black', linewidths=0.8, zorder=5)
ax.set_title(f'3D + Peak locations', fontweight='bold')

plt.suptitle(f'{sample_base} — GT={GT_TILLERS.get(sample_base,"?")} | '
             f'Pred={len(peak_coords)} (Height Map method)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2b_heightmap_example.png', dpi=200, bbox_inches='tight')
plt.show()





import shutil

# Tham số: ('tên_file_sau_khi_nén', 'định_dạng', 'thư_mục_nguồn')
shutil.make_archive('output_files', 'zip', '/kaggle/working')

print("Đã zip xong toàn bộ file!")


import os

file_path = '/kaggle/working/output_files.zip'

if os.path.exists(file_path):
    os.remove(file_path)
    print("Đã xóa file thành công, bộ nhớ đã được giải phóng!")
else:
    print("Không tìm thấy file.")
